# 01 — ESMFold2 smoke test (thí nghiệm khả thi, chạy MỘT lần)

**Notebook này trả lời 5 câu hỏi quyết định.** Không phải pipeline. Chỉ đo.

| # | Câu hỏi | Cách đo |
|---|---|---|
| 1 | ESMFold2 (6.58 B, lưu F32 = 26.3 GB) có **load và chạy nổi** trên L4 23 GB không? | load `dtype=torch.bfloat16` + `esmc_precision="fp8"`, đo `max_memory_allocated()` |
| 2 | Một phức IL-7Rα + binder tốn **bao nhiêu VRAM / bao nhiêu phút** ở đúng cấu hình cửa sàng? | fold thật, bấm giờ |
| 3 | **Trần số residue** là bao nhiêu trước khi OOM? | leo thang 235 → 775 aa, bắt OOM sạch |
| 4 | Đường **MSA** (MSA cho target, không MSA cho binder) có chạy không? | `ProteinInput(..., msa=MSA.from_a3m(...))` |
| 5 | Lấy được `result.pae` và đẩy sang **ipSAE** đúng thứ tự token không? | tự ghi PDB theo thứ tự token + audit permutation |

**Yêu cầu runtime:** GPU **L4** (hoặc tốt hơn). T4 16 GB **không chạy nổi** — notebook sẽ dừng ngay ở cell 1.
Cần Google Drive còn **≥ 32 GB** (weights 26.7 GB).

**Thời gian dự kiến:**
- Lần chạy đầu (phải tải 26.7 GB weights): **55–90 phút** → **≈ 4.5–7 compute unit**
- Lần chạy sau (weights đã cache trên Drive): **35–55 phút** → **≈ 3–4.5 CU**

**Chặn cứng:** notebook tự `raise SystemExit` nếu vượt `HARD_CAP_MIN` phút (mặc định 110 phút ≈ 8.8 CU).
Nó in số CU đã đốt sau **mỗi** bước — nếu thấy vô lý thì bấm dừng, kết quả các bước trước đã ghi xuống Drive rồi.

**Nguyên tắc chống mất dữ liệu:** sau mỗi lần fold, kết quả được ghi ngay vào
`/content/drive/MyDrive/proteinAI/out/01_esmfold2_smoke_partial.json`.
Runtime chết giữa đường thì cứ gửi file đó.

**API đã được verify bằng source thật** (`esm==3.4.3` wheel + `config.json` của `biohub/ESMFold2`
+ `ipsae.py` v1). Xem `docs/05-CODE-NOTES.md`.

## Cell 1 — TIỀN KIỂM (chạy trước khi cài gì)

Kiểm GPU / VRAM / Drive / dung lượng. Thiếu điều kiện → dừng ngay kèm lời nhắc cụ thể.

In [ ]:
# ============================== CELL 1: PRE-FLIGHT ==============================
import os, sys, time, json, shutil, subprocess, platform

# Must be set BEFORE torch is imported. Reduces fragmentation, which matters a lot
# for the OOM-ceiling probe. Recorded in the report because it changes the ceiling.
ALLOC_CONF = "expandable_segments:True"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", ALLOC_CONF)

# ---- knobs (đổi ở đây, đừng sửa rải rác trong notebook) ----
HARD_CAP_MIN       = 110      # chặn cứng: vượt là dừng
CU_PER_HOUR        = 4.8      # L4 Colab Pro
ESMC_PRECISION     = "fp8"    # "fp8" (ít VRAM nhất, L4 cc 8.9 hỗ trợ) | "bf16" | "fp32"
MODEL_DTYPE        = "bfloat16"
GATE_NUM_LOOPS     = 20       # config.json của biohub/ESMFold2: num_loops=20
GATE_SAMPLING_STEPS= 200      # mặc định của builder.fold()
PROBE_NUM_LOOPS    = 4        # probe trần VRAM: VRAM gần như độc lập với số loop
PROBE_SAMPLING_STEPS = 20
# mặc định của builder.fold() trong esm 3.4.3 — viết ra để báo cáo ghi lại được.
# LƯU Ý: lm_dropout=0.3 là NGẪU NHIÊN (seed hoá được), còn config.json của checkpoint
# ghi lm_encoder.lm_dropout=0.25 → đây là một thông số PHẢI hỏi organizer.
LM_DROPOUT         = 0.3
MSA_MAX_DEPTH      = 1024
MSA_COLUMN_MASK_RATE = 0.1
MSA_TIMEOUT_S      = 420      # quá thời gian này thì bỏ MSA thật, dùng a3m query-only
STAGE_TO_LOCAL     = True     # copy weights Drive -> /content rồi mới load (mmap ổn định hơn)

T0 = time.time()

def elapsed_min():
    return (time.time() - T0) / 60.0

def cu_burned():
    return elapsed_min() / 60.0 * CU_PER_HOUR

def budget(step=""):
    """In tiến độ + CU đã đốt. Dừng cứng nếu vượt HARD_CAP_MIN."""
    m, cu = elapsed_min(), cu_burned()
    bar = "!!!" if m > 0.8 * HARD_CAP_MIN else ("!" if m > 0.5 * HARD_CAP_MIN else "")
    print(f"[NGÂN SÁCH]{bar} {m:6.1f} phút đã chạy · ước {cu:4.2f} CU đã đốt "
          f"· trần {HARD_CAP_MIN} phút ({HARD_CAP_MIN/60*CU_PER_HOUR:.1f} CU) {step}")
    if m > HARD_CAP_MIN:
        raise SystemExit(
            f"DỪNG CỨNG: notebook đã chạy {m:.0f} phút (> HARD_CAP_MIN={HARD_CAP_MIN}). "
            f"Ước đã đốt {cu:.1f} CU. Kết quả từng phần đã nằm ở "
            f"out/01_esmfold2_smoke_partial.json trên Drive — gửi file đó cho Claude."
        )

ENV = {"python": platform.python_version(), "platform": platform.platform()}

# ---- GPU ----
try:
    import torch
except Exception as e:
    raise SystemExit(f"DỪNG: không import được torch ({e}). Đây không phải runtime Colab bình thường.")

if not torch.cuda.is_available():
    raise SystemExit(
        "DỪNG: KHÔNG CÓ GPU. Vào Runtime > Change runtime type > Hardware accelerator = GPU (L4), "
        "rồi Run all lại từ đầu. Đừng chạy tiếp các cell dưới — chúng sẽ tải 26.7 GB vô ích."
    )

gpu_name  = torch.cuda.get_device_name(0)
cc        = torch.cuda.get_device_capability(0)
free_b, total_b = torch.cuda.mem_get_info(0)
vram_total_gb = total_b / 1024**3
vram_free_gb  = free_b / 1024**3
ENV.update(torch_version=torch.__version__, gpu=gpu_name, compute_capability=list(cc),
           vram_total_gb=round(vram_total_gb, 2), vram_free_gb=round(vram_free_gb, 2),
           alloc_conf=os.environ.get("PYTORCH_CUDA_ALLOC_CONF"))
print(f"GPU            : {gpu_name}  (compute capability {cc[0]}.{cc[1]})")
print(f"VRAM           : {vram_total_gb:.2f} GB tổng · {vram_free_gb:.2f} GB còn trống")
print(f"torch          : {torch.__version__}")
print(f"python         : {platform.python_version()}")

if vram_total_gb < 20.0:
    raise SystemExit(
        f"DỪNG: GPU '{gpu_name}' chỉ có {vram_total_gb:.1f} GB VRAM. ESMFold2 cần ≥ 20 GB "
        f"(weights bf16+fp8 ≈ 7 GB, còn lại cho activation). "
        f"Nếu đang là T4 16 GB: Runtime > Change runtime type > chọn L4. "
        f"KHÔNG chạy tiếp — notebook sẽ OOM sau khi đã tải 26.7 GB weights."
    )
if cc[0] * 10 + cc[1] < 89 and ESMC_PRECISION == "fp8":
    print(f"CẢNH BÁO: GPU này (cc {cc[0]}.{cc[1]}) không có fp8 GEMM (cần ≥ 8.9). "
          f"Weight vẫn ở fp8 (tiết kiệm VRAM) nhưng mỗi matmul phải upcast → chậm hơn.")

# ---- Drive ----
try:
    from google.colab import drive  # noqa
    drive.mount("/content/drive")
except Exception as e:
    raise SystemExit(
        f"DỪNG: không mount được Google Drive ({e}). Notebook này BẮT BUỘC cần Drive "
        f"để cache 26.7 GB weights — nếu không, mỗi lần chạy lại phải tải lại từ đầu."
    )

ROOT    = "/content/drive/MyDrive/proteinAI"
D_W     = f"{ROOT}/weights/ESMFold2"
D_DATA  = f"{ROOT}/data"
D_RUNS  = f"{ROOT}/runs/01_smoke"
D_OUT   = f"{ROOT}/out"
D_LOGS  = f"{ROOT}/logs"
for d in (D_W, D_DATA, D_RUNS, D_OUT, D_LOGS):
    os.makedirs(d, exist_ok=True)

L_W = "/content/esmfold2_weights"   # staging cục bộ

du_drive   = shutil.disk_usage(ROOT)
du_content = shutil.disk_usage("/content")
drive_free_gb   = du_drive.free / 1024**3
content_free_gb = du_content.free / 1024**3
ENV.update(drive_free_gb=round(drive_free_gb, 1), content_free_gb=round(content_free_gb, 1))
print(f"Drive          : còn {drive_free_gb:.1f} GB trống  ({ROOT})")
print(f"/content       : còn {content_free_gb:.1f} GB trống")

WEIGHTS_BYTES = 26_737_068_175   # tổng chính xác, lấy từ HF API (xem cell 3)
already_cached = os.path.exists(f"{D_W}/config.json")
need_drive_gb  = 2.0 if already_cached else WEIGHTS_BYTES / 1024**3 + 2.0
if drive_free_gb < need_drive_gb:
    raise SystemExit(
        f"DỪNG: Drive chỉ còn {drive_free_gb:.1f} GB, cần ≥ {need_drive_gb:.1f} GB "
        f"(weights ESMFold2 = 26.7 GB + chỗ cho output). Dọn Drive rồi chạy lại."
    )
if STAGE_TO_LOCAL and content_free_gb < 30.0:
    print(f"CẢNH BÁO: /content chỉ còn {content_free_gb:.1f} GB → tắt STAGE_TO_LOCAL, "
          f"sẽ load weights trực tiếp từ Drive (chậm hơn, rủi ro mmap cao hơn).")
    STAGE_TO_LOCAL = False

# ---- CPU / RAM ----
try:
    n_cpu = os.cpu_count()
    with open("/proc/meminfo") as f:
        mem_total_gb = int([l for l in f if l.startswith("MemTotal")][0].split()[1]) / 1024**2
except Exception:
    n_cpu, mem_total_gb = None, None
ENV.update(cpu_count=n_cpu, ram_total_gb=None if mem_total_gb is None else round(mem_total_gb, 1))
print(f"CPU / RAM      : {n_cpu} core · {mem_total_gb:.0f} GB RAM")
if mem_total_gb is not None and mem_total_gb < 40:
    print("CẢNH BÁO: < 40 GB RAM. Khi load, checkpoint F32 (26.3 GB) đi qua RAM trước khi "
          "cast sang bf16 → có thể bị kill. Nếu runtime chết lúc load, báo lại cho Claude.")

# ---- báo cáo từng phần: ghi ngay, ghi liên tục ----
REPORT = {
    "notebook": "01_esmfold2_smoke_test",
    "schema": 1,
    "started_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "env": ENV,
    "knobs": {
        "esmc_precision": ESMC_PRECISION, "model_dtype": MODEL_DTYPE,
        "gate_num_loops": GATE_NUM_LOOPS, "gate_num_sampling_steps": GATE_SAMPLING_STEPS,
        "probe_num_loops": PROBE_NUM_LOOPS, "probe_num_sampling_steps": PROBE_SAMPLING_STEPS,
        "lm_dropout": LM_DROPOUT, "msa_max_depth": MSA_MAX_DEPTH,
        "msa_column_mask_rate": MSA_COLUMN_MASK_RATE,
        "hard_cap_min": HARD_CAP_MIN, "stage_to_local": STAGE_TO_LOCAL,
        "alloc_conf": os.environ.get("PYTORCH_CUDA_ALLOC_CONF"),
    },
    "steps": {},
    "folds": [],
    "ceiling_probe": [],
    "errors": [],
    "manual_fill": {
        "compute_units_remaining_before_run": "<<ĐIỀN TAY>>",
        "compute_units_remaining_after_run": "<<ĐIỀN TAY>>",
    },
}
PARTIAL = f"{D_OUT}/01_esmfold2_smoke_partial.json"

def save_partial():
    REPORT["elapsed_min"] = round(elapsed_min(), 2)
    REPORT["cu_estimate"] = round(cu_burned(), 3)
    tmp = PARTIAL + ".tmp"
    with open(tmp, "w") as f:
        json.dump(REPORT, f, indent=2, default=str)
    os.replace(tmp, PARTIAL)

save_partial()
print(f"\nBáo cáo từng phần ghi liên tục vào: {PARTIAL}")
print("TIỀN KIỂM: OK — chạy tiếp được.")
budget("(cell 1 xong)")

## Cell 2 — cài `esm==3.4.3`

`esm` yêu cầu `torch>=2.11` và `transformers>=5.16`. Runtime đã có torch 2.14 nên pip **không**
được phép thay torch. Nếu nó thay → CUDA có thể lệch, và ta vừa mất ~5 phút tải 4.6 GB wheel vô ích,
nên cell này dừng cứng và yêu cầu restart runtime.

In [ ]:
# ============================== CELL 2: INSTALL esm ==============================
budget("(bắt đầu cell 2)")
t = time.time()
torch_before = torch.__version__

try:
    import esm as _esm_probe
    have = getattr(_esm_probe, "__version__", "unknown")
except Exception:
    have = None

if have is None:
    print("Đang cài esm==3.4.3 ... (1–4 phút)")
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "esm==3.4.3"],
        capture_output=True, text=True,
    )
    print(r.stdout[-3000:]);  print(r.stderr[-4000:])
    if r.returncode != 0:
        REPORT["errors"].append({"step": "pip_install_esm", "returncode": r.returncode,
                                 "stderr": r.stderr[-4000:]})
        save_partial()
        raise SystemExit("DỪNG: pip install esm==3.4.3 thất bại. Copy nguyên output đỏ ở trên gửi Claude.")
else:
    print(f"esm đã có sẵn (version {have}) — bỏ qua pip.")

import importlib
importlib.invalidate_caches()   # pip vừa ghi vào site-packages trong cùng process
install_s = time.time() - t

# pip có thể đã thay torch -> phải restart runtime, không thể import lại trong cùng process
r2 = subprocess.run([sys.executable, "-c", "import torch;print(torch.__version__)"],
                    capture_output=True, text=True)
torch_after = r2.stdout.strip()
print(f"\ntorch trước pip : {torch_before}")
print(f"torch sau  pip : {torch_after}")
if torch_after and torch_after != torch_before:
    REPORT["errors"].append({"step": "pip_install_esm",
                             "msg": f"pip replaced torch {torch_before} -> {torch_after}"})
    save_partial()
    raise SystemExit(
        f"DỪNG: pip đã thay torch {torch_before} -> {torch_after}. "
        f"Phải Runtime > Restart session rồi Run all lại (weights đã cache nên lần 2 nhanh hơn). "
        f"Báo lại cho Claude nếu lặp lại."
    )

import esm
from esm.models.esmfold2 import (
    EsmFold2Model, ESMFold2InputBuilder, ProteinInput, StructurePredictionInput,
    ESMFOLD2_HF_REPO,
)
from esm.utils.msa import MSA
import huggingface_hub

print(f"esm            : {getattr(esm, '__version__', 'unknown')}")
print(f"huggingface_hub: {huggingface_hub.__version__}")
print(f"HF repo        : {ESMFOLD2_HF_REPO}")

REPORT["steps"]["install"] = {
    "seconds": round(install_s, 1),
    "esm_version": getattr(esm, "__version__", "unknown"),
    "torch_before": torch_before, "torch_after": torch_after,
    "esmfold2_hf_repo": ESMFOLD2_HF_REPO,
}
save_partial()
budget("(cell 2 xong)")

## Cell 3 — cache 26.7 GB weights lên Drive (tải MỘT lần, dùng mãi)

Manifest dưới đây là **kích thước byte chính xác** lấy từ HF API của `biohub/ESMFold2`
(commit `69869f7`). Một file lệch 1 byte = cache hỏng → cell này tải lại đúng file đó.
Đây là lá chắn chống "cache tải nửa đường" làm sai mọi số đo về sau.

`ccd.pkl` (417 MB) **không** nằm trong `snapshot_download` của `esm`
(nó chỉ lấy `*.json` + `*.safetensors`) nên ta phải tải tay rồi truyền qua
`ESMFold2InputBuilder(ccd_cache=...)`.

In [ ]:
# ============================== CELL 3: WEIGHTS CACHE ==============================
budget("(bắt đầu cell 3)")
from huggingface_hub import hf_hub_download

# Byte-exact manifest, nguồn: https://huggingface.co/api/models/biohub/ESMFold2?blobs=true
MANIFEST = {
    "config.json":                        8214,
    "model.safetensors.index.json":       245594,
    "model-00001-of-00006.safetensors":   4955672788,
    "model-00002-of-00006.safetensors":   4971194264,
    "model-00003-of-00006.safetensors":   4997460576,
    "model-00004-of-00006.safetensors":   4979069192,
    "model-00005-of-00006.safetensors":   4934556472,
    "model-00006-of-00006.safetensors":   1481396616,
    "ccd.pkl":                            417306584,
}
MANIFEST_TOTAL = sum(MANIFEST.values())
print(f"Cần tổng {MANIFEST_TOTAL/1024**3:.2f} GB / {len(MANIFEST)} file.")

def audit(d):
    """Trả về (ok_list, bad_list). bad = thiếu hoặc sai kích thước."""
    ok, bad = [], []
    for fn, size in MANIFEST.items():
        p = os.path.join(d, fn)
        if not os.path.exists(p):
            bad.append((fn, "missing", 0, size))
        else:
            got = os.path.getsize(p)
            (ok if got == size else bad).append(
                (fn, "ok" if got == size else "size-mismatch", got, size))
    return ok, bad

def safetensors_header_sane(p):
    """8 byte đầu = độ dài header JSON. Rẻ, bắt được file cắt cụt ở đầu."""
    try:
        with open(p, "rb") as f:
            n = int.from_bytes(f.read(8), "little")
        return 0 < n < 50_000_000 and n + 8 <= os.path.getsize(p)
    except Exception:
        return False

ok, bad = audit(D_W)
print(f"\nCache trên Drive ({D_W}): {len(ok)}/{len(MANIFEST)} file đúng kích thước.")
for fn, why, got, want in bad:
    print(f"  CẦN TẢI  {fn:38s} {why:14s} got={got} want={want}")

dl_seconds, dl_bytes = 0.0, 0
if bad:
    # Tải xuống /content (nhanh) rồi copy sang Drive (một lần ghi duy nhất).
    stage = "/content/hf_dl"; os.makedirs(stage, exist_ok=True)
    t = time.time()
    for i, (fn, why, got, want) in enumerate(bad, 1):
        print(f"[{i}/{len(bad)}] tải {fn} ({want/1024**3:.2f} GB) ...", flush=True)
        ts = time.time()
        p = hf_hub_download(repo_id="biohub/ESMFold2", filename=fn,
                            local_dir=stage, force_download=(why == "size-mismatch"))
        got2 = os.path.getsize(p)
        if got2 != want:   # một lần thử lại, lần này cưỡng chế tải mới
            print(f"      kích thước lệch ({got2} != {want}) -> force_download, thử lại 1 lần")
            try:
                os.remove(p)
            except OSError:
                pass
            p = hf_hub_download(repo_id="biohub/ESMFold2", filename=fn,
                                local_dir=stage, force_download=True)
            got2 = os.path.getsize(p)
        if got2 != want:
            REPORT["errors"].append({"step": "download", "file": fn, "got": got2, "want": want})
            save_partial()
            raise SystemExit(f"DỪNG: {fn} tải về {got2} byte, cần {want}. Mạng lỗi — chạy lại cell 3.")
        print(f"      xong trong {time.time()-ts:.0f}s → copy sang Drive ...", flush=True)
        shutil.copy2(p, os.path.join(D_W, fn))
        os.remove(p)
        dl_bytes += want
        print(f"      tổng {dl_bytes/1024**3:.2f} GB · {elapsed_min():.1f} phút", flush=True)
        budget(f"(đang tải weights {i}/{len(bad)})")
    dl_seconds = time.time() - t
    shutil.rmtree(stage, ignore_errors=True)

ok, bad = audit(D_W)
if bad:
    REPORT["errors"].append({"step": "verify_cache", "bad": bad})
    save_partial()
    raise SystemExit(f"DỪNG: cache vẫn sai sau khi tải: {bad}")
print(f"\nCache Drive ĐẦY ĐỦ ({len(ok)}/{len(MANIFEST)} file, kích thước khớp từng byte).")

# kiểm tra index.json trỏ tới đúng các shard đang có
with open(os.path.join(D_W, "model.safetensors.index.json")) as f:
    shards = sorted(set(json.load(f)["weight_map"].values()))
missing = [s for s in shards if s not in MANIFEST]
if missing:
    REPORT["errors"].append({"step": "verify_cache", "index_refs_missing": missing})
    save_partial()
    raise SystemExit(f"DỪNG: index.json trỏ tới shard không có trong manifest: {missing}")
print(f"index.json trỏ tới {len(shards)} shard — tất cả đều có.")

# staging sang /content: mmap trên Google Drive FUSE không đáng tin khi đọc 26 GB
MODEL_DIR = D_W
stage_seconds = 0.0
if STAGE_TO_LOCAL:
    os.makedirs(L_W, exist_ok=True)
    ok_l, bad_l = audit(L_W)
    if bad_l:
        print(f"\nCopy weights Drive → {L_W} ({MANIFEST_TOTAL/1024**3:.1f} GB, ~2–4 phút) ...")
        t = time.time()
        for fn, why, got, want in bad_l:
            print(f"  {fn} ...", flush=True)
            shutil.copy2(os.path.join(D_W, fn), os.path.join(L_W, fn))
        stage_seconds = time.time() - t
        ok_l, bad_l = audit(L_W)
        if bad_l:
            REPORT["errors"].append({"step": "stage_local", "bad": bad_l})
            save_partial()
            raise SystemExit(f"DỪNG: copy sang /content bị lỗi: {bad_l}")
        print(f"  xong trong {stage_seconds:.0f}s")
    else:
        print(f"\n{L_W} đã có sẵn đủ weights — bỏ qua copy.")
    MODEL_DIR = L_W

bad_hdr = [fn for fn in MANIFEST if fn.endswith(".safetensors")
           and not safetensors_header_sane(os.path.join(MODEL_DIR, fn))]
if bad_hdr:
    REPORT["errors"].append({"step": "verify_cache", "bad_safetensors_header": bad_hdr})
    save_partial()
    raise SystemExit(f"DỪNG: header safetensors không hợp lệ: {bad_hdr}")

CCD_DIR = MODEL_DIR  # load_ccd(cache_dir) đọc <cache_dir>/ccd.pkl
print(f"\nMODEL_DIR = {MODEL_DIR}")
REPORT["steps"]["weights"] = {
    "cache_dir_drive": D_W, "model_dir_used": MODEL_DIR,
    "cache_hit": dl_bytes == 0,
    "download_seconds": round(dl_seconds, 1),
    "download_gb": round(dl_bytes / 1024**3, 3),
    "stage_to_local_seconds": round(stage_seconds, 1),
    "manifest_total_gb": round(MANIFEST_TOTAL / 1024**3, 3),
    "files_verified": len(ok),
}
save_partial()
budget("(cell 3 xong)")

## Cell 4 — load model (bf16 + ESMC fp8), đo VRAM weights

**Đây là câu hỏi 1.** Checkpoint lưu F32 → 26.3 GB, không nhét nổi vào 21.5 GB.
`EsmFold2Model.from_pretrained` **có** tham số `dtype` (đã đọc source `esm/models/esmfold2/model.py`),
và nó cast trên CPU *trước* khi move sang GPU, nên fp32 không bao giờ chạm GPU.

Ước tính: ESMC-6B ở fp8 ≈ 6.4 GB + trunk bf16 ≈ 0.5 GB → **≈ 7 GB weights trên GPU**,
còn ~14 GB cho activation. Cell này đo con số thật.

Hai thứ cell này cũng kiểm tra, vì cả hai đều âm thầm làm sai điểm nếu thiếu:
- `model.msa_encoder is not None` → nếu `None` thì MSA bị **bỏ qua không báo lỗi** (đúng bệnh của checkpoint `-Fast`).
- `model._offload_esmc` → cờ offload ESMC sang CPU sau khi tính LM features. Có trong `esm` 3.4.3
  (release model, không chỉ riêng đường distributed) → đòn bẩy VRAM chính cho probe trần.

In [ ]:
# ============================== CELL 4: LOAD MODEL ==============================
budget("(bắt đầu cell 4)")
import gc, warnings

def vram():
    return (torch.cuda.memory_allocated() / 1024**3,
            torch.cuda.max_memory_allocated() / 1024**3,
            torch.cuda.max_memory_reserved() / 1024**3)

def rss_gb():
    try:
        with open("/proc/self/status") as f:
            for l in f:
                if l.startswith("VmRSS"):
                    return int(l.split()[1]) / 1024**2
    except Exception:
        return None

torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
DTYPE = getattr(torch, MODEL_DTYPE)
print(f"Load {MODEL_DIR}\n  dtype={MODEL_DTYPE}  esmc_precision={ESMC_PRECISION}")
print("  (đọc 26.3 GB safetensors → cast bf16 trên CPU → quantize ESMC → move GPU; 4–10 phút)")

t = time.time()
model = EsmFold2Model.from_pretrained(
    MODEL_DIR,
    device="cuda",
    dtype=DTYPE,
    esmc_precision=ESMC_PRECISION,   # "fp8" | "bf16" | "fp32"
    load_esmc=True,
).eval()
load_s = time.time() - t

KERNEL_BACKEND = "fused"             # Triton, bit-exact với reference path
try:
    model.set_kernel_backend("fused")
except Exception as e:
    KERNEL_BACKEND = None
    model.set_kernel_backend(None)
    print(f"CẢNH BÁO: không bật được Triton kernel 'fused' ({type(e).__name__}: {e}) "
          f"-> dùng reference path (chậm hơn, kết quả như nhau).")

alloc, peak, reserved = vram()
n_params = sum(p.numel() for p in model.parameters())
n_buffers = sum(b.numel() for b in model.buffers())
has_msa  = model.msa_encoder is not None
has_offl = hasattr(model, "_offload_esmc")
trunk_dtype = str(next(model.folding_trunk.parameters()).dtype)

print(f"\nLoad xong trong {load_s/60:.1f} phút")
print(f"  VRAM weights   : {alloc:.2f} GB (peak trong lúc load {peak:.2f} GB, reserved {reserved:.2f} GB)")
print(f"  VRAM còn trống : {torch.cuda.mem_get_info(0)[0]/1024**3:.2f} GB")
print(f"  RAM (RSS)      : {rss_gb():.1f} GB")
print(f"  params         : {n_params:,}  (+ {n_buffers:,} buffer element, fp8 nằm trong buffer)")
print(f"  trunk dtype    : {trunk_dtype}")
print(f"  num_loops cfg  : {model.config.num_loops}")
print(f"  MSA encoder    : {'CÓ' if has_msa else 'KHÔNG'}  <- {'ok, đọc được MSA' if has_msa else 'MSA SẼ BỊ BỎ QUA ÂM THẦM!'}")
print(f"  _offload_esmc  : {'có cờ' if has_offl else 'không có cờ'} (giá trị hiện tại: {getattr(model, '_offload_esmc', None)})")

if not has_msa:
    print("\n!!! CẢNH BÁO NẶNG: checkpoint này không có MSA encoder. Mọi MSA truyền vào sẽ bị "
          "bỏ qua không báo lỗi → không tái dựng được cửa sàng. Báo ngay cho Claude.")

t = time.time()
builder = ESMFold2InputBuilder(ccd_cache=CCD_DIR)   # đọc <CCD_DIR>/ccd.pkl, không tải mạng
ccd_s = time.time() - t
print(f"\nESMFold2InputBuilder: OK (ccd.pkl 417 MB đọc từ cache trong {ccd_s:.0f}s).")

REPORT["steps"]["load_model"] = {
    "seconds": round(load_s, 1),
    "ccd_load_seconds": round(ccd_s, 1),
    "kernel_backend_effective": KERNEL_BACKEND,
    "vram_weights_gb": round(alloc, 3),
    "vram_peak_during_load_gb": round(peak, 3),
    "vram_reserved_gb": round(reserved, 3),
    "vram_free_after_load_gb": round(torch.cuda.mem_get_info(0)[0] / 1024**3, 3),
    "ram_rss_gb": rss_gb(),
    "n_params": n_params, "n_buffer_elements": n_buffers,
    "trunk_dtype": trunk_dtype,
    "config_num_loops": model.config.num_loops,
    "msa_encoder_present": bool(has_msa),
    "offload_esmc_flag_present": bool(has_offl),
}
save_partial()
budget("(cell 4 xong — model đã nằm trên GPU)")

## Cell 5 — sequence: target thật + binder dương tính + binder âm tính

- **IL-7Rα** `P16871` Glu21–Asp239 = 219 aa (ectodomain đầy đủ, đúng range cuộc thi)
- **TL1A** `O95150` Leu72–Leu251 = 180 aa (dùng cho probe trần: 1/2/3/4 protomer)
- **Binder dương tính**: minibinder de novo 55 aa đã có structure (PDB 7S5B / 7OPB chain D–F,
  Cao et al. Nature 2022), affinity picomolar. **Đây là positive control đúng nghĩa**: nếu
  ESMFold2+ipSAE chấm một binder đã validate thật mà điểm thấp, thì *setup chấm điểm của ta sai*,
  và phải biết điều đó trước khi tin bất kỳ design nào của mình.
- **Binder âm tính**: chính 55-mer đó bị **đảo thứ tự** (seed cố định) — giữ nguyên thành phần
  amino acid, phá sạch cấu trúc. Khoảng cách điểm giữa hai cái mới nói được pipeline có **phân biệt** nổi hay không.

In [ ]:
# ============================== CELL 5: SEQUENCES ==============================
budget("(bắt đầu cell 5)")
import random

# --- target thật, copy nguyên văn từ data/*.fasta của repo (notebook tự chứa) ---
IL7RA = (
    "ESGYAQNGDLEDAELDDYSFSCYSQLEVNGSQHSLTCAFEDPDVNITNLEFEICGALVEVKCLNFRKLQEIYFIETK"
    "KFLLIGKSNICVKVGEKSLTCKKIDLTTIVKPEAPFDLSVVYREGANDFVVTFNTSHLQKKYVKVLMHDVAYRQEKD"
    "ENKWTHVNLSSTKLTLLQRKLQPAAMYEIKVRSIPDHYFKGFWSEWSPSYYFRTPEINNSSGEMD"
)
TL1A = (
    "LKGQEFAPSHQQVYAPLRADGDKPRAHLTVVRQTPTQHFKNQFPALHWEHELGLAFTKNRMNYTNKFLLIPESGDYF"
    "IYSQVTFRGMTSECSEIRQAGRPNKPDSITVVITKVTDSYPEPTQLLMGTKSVCEVGSNWFQPIYLGAMFSLQEGDK"
    "LMVNVSDISLVDYTKEDKTFFGAFLL"
)
# PDB 7S5B / 7OPB chain D-E-F — de novo IL-7Ra minibinder, 55 aa, Cao et al. Nature 2022
BINDER_POS = "SVIEKLRKLEKQARKQGDEVLVMLARMVLEYLEKGWVSEEDADESADRIEEVLKK"

assert len(IL7RA) == 219, len(IL7RA)
assert len(TL1A) == 180, len(TL1A)
assert len(BINDER_POS) == 55, len(BINDER_POS)
assert IL7RA.startswith("ESG") and IL7RA.endswith("GEMD")
assert TL1A.startswith("LKGQ") and TL1A.endswith("AFLL")

# negative control: đảo thứ tự, seed cố định -> tái lập được
_rng = random.Random(20261111)
_chars = list(BINDER_POS); _rng.shuffle(_chars)
BINDER_NEG = "".join(_chars)
assert sorted(BINDER_NEG) == sorted(BINDER_POS) and BINDER_NEG != BINDER_POS

# binder 100 aa (chỉ dùng cho probe trần VRAM, không phải design)
BINDER_100 = (BINDER_POS + BINDER_POS)[:100]

print(f"IL-7Ra     {len(IL7RA):4d} aa  {IL7RA[:30]}...{IL7RA[-10:]}")
print(f"TL1A       {len(TL1A):4d} aa  {TL1A[:30]}...{TL1A[-10:]}")
print(f"binder (+) {len(BINDER_POS):4d} aa  {BINDER_POS}")
print(f"binder (-) {len(BINDER_NEG):4d} aa  {BINDER_NEG}   <- scrambled, seed 20261111")
print(f"binder100  {len(BINDER_100):4d} aa  (chỉ để probe VRAM)")
print(f"\nPhức chính: IL-7Ra + binder = {len(IL7RA)+len(BINDER_POS)} residue")

REPORT["steps"]["sequences"] = {
    "il7ra_len": len(IL7RA), "tl1a_len": len(TL1A),
    "binder_pos": BINDER_POS, "binder_pos_source": "PDB 7S5B / 7OPB chain D-F, Cao et al. Nature 2022",
    "binder_neg": BINDER_NEG, "binder_neg_rule": "random.Random(20261111).shuffle of binder_pos",
    "binder_100_len": len(BINDER_100),
    "main_complex_total_residues": len(IL7RA) + len(BINDER_POS),
}
save_partial()
budget("(cell 5 xong)")

## Cell 6 — MSA cho target (câu hỏi 4)

Cửa sàng dùng **MSA cho target, không MSA cho binder**. ESMFold2 nhận MSA theo từng chain:
`ProteinInput(..., msa=MSA.from_a3m(path=...))`.

Cell này thử lấy MSA **thật** từ ColabFold MSA API (`https://api.colabfold.com`, cùng server mà
`colabfold_batch` dùng; protocol đã đọc từ source `sokrypton/ColabFold`). Có timeout cứng
`MSA_TIMEOUT_S`. Nếu server chậm/lỗi → **fallback sang a3m chỉ chứa query** và ghi rõ
`msa_source="query_only_FALLBACK"`.

> **Đọc kỹ chỗ này:** a3m query-only vẫn **chứng minh được đường code MSA chạy** (encoder nhận
> đúng shape, không crash), nhưng **không** phải tái dựng cửa sàng — depth 1 ≈ không có MSA.
> Báo cáo sẽ ghi `msa_depth_realistic: false` để không ai nhầm.

In [ ]:
# ============================== CELL 6: MSA ==============================
budget("(bắt đầu cell 6)")
import tarfile, io as _io, urllib.request, urllib.parse, urllib.error

from esm.utils.msa.msa import is_a3m_insertion

A3M_DIR = f"{D_DATA}/msa"; os.makedirs(A3M_DIR, exist_ok=True)
A3M_IL7RA = f"{A3M_DIR}/IL7Ra_P16871_E21-D239.uniref.a3m"
HOST = "https://api.colabfold.com"
UA = "proteinAI-changjiangcup/0.1 (smoke test)"

def _post(url, data, timeout=30):
    body = urllib.parse.urlencode(data).encode()
    req = urllib.request.Request(url, data=body, headers={"User-Agent": UA})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read().decode())

def _get_json(url, timeout=30):
    req = urllib.request.Request(url, headers={"User-Agent": UA})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read().decode())

def _get_bytes(url, timeout=300):
    req = urllib.request.Request(url, headers={"User-Agent": UA})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

def fetch_colabfold_a3m(seq, out_path, deadline_s):
    """ColabFold MSA API: POST /ticket/msa -> poll /ticket/<id> -> GET /result/download/<id>.
    Trả về (ok, info)."""
    t0 = time.time()
    out = _post(f"{HOST}/ticket/msa", {"q": f">101\n{seq}\n", "mode": "env"})
    st = out.get("status")
    if st in ("ERROR", "MAINTENANCE"):
        return False, {"stage": "submit", "status": st}
    tid = out.get("id")
    if not tid:
        return False, {"stage": "submit", "status": st, "raw": str(out)[:300]}
    print(f"  ticket {tid} ({st}) — chờ server MSA ...", flush=True)
    while st in ("UNKNOWN", "RUNNING", "PENDING", "RATELIMIT"):
        if time.time() - t0 > deadline_s:
            return False, {"stage": "poll", "status": st, "timeout_s": deadline_s}
        time.sleep(8)
        st = _get_json(f"{HOST}/ticket/{tid}").get("status")
        print(f"    ... {st}  ({time.time()-t0:.0f}s)", flush=True)
    if st != "COMPLETE":
        return False, {"stage": "poll", "status": st}
    blob = _get_bytes(f"{HOST}/result/download/{tid}", timeout=max(60, int(deadline_s)))
    with tarfile.open(fileobj=_io.BytesIO(blob)) as tar:
        names = tar.getnames()
        member = next((n for n in names if n.endswith("uniref.a3m")), None)
        if member is None:
            return False, {"stage": "extract", "members": names}
        data = tar.extractfile(member).read()
    with open(out_path, "wb") as f:
        f.write(data)
    return True, {"ticket": tid, "seconds": round(time.time() - t0, 1), "members": names}

def write_query_only_a3m(seq, path, name):
    with open(path, "w") as f:
        f.write(f">{name}\n{seq}\n")

msa_info, msa_source = {}, None
if os.path.exists(A3M_IL7RA) and os.path.getsize(A3M_IL7RA) > 1000:
    print(f"a3m đã cache: {A3M_IL7RA}")
    msa_source = "colabfold_uniref_cached"
else:
    print(f"Gọi ColabFold MSA API cho IL-7Ra (219 aa), timeout {MSA_TIMEOUT_S}s ...")
    try:
        ok, info = fetch_colabfold_a3m(IL7RA, A3M_IL7RA, MSA_TIMEOUT_S)
        msa_info = info
        msa_source = "colabfold_uniref" if ok else None
        if not ok:
            print(f"  THẤT BẠI: {info}")
    except Exception as e:
        msa_info = {"exception": f"{type(e).__name__}: {e}"}
        print(f"  THẤT BẠI: {msa_info}")

if msa_source is None:
    print("  -> fallback: a3m CHỈ CHỨA QUERY (chứng minh đường code, KHÔNG phải MSA thật)")
    A3M_IL7RA = f"{A3M_DIR}/IL7Ra_query_only.a3m"
    write_query_only_a3m(IL7RA, A3M_IL7RA, "IL7Ra_query_only")
    msa_source = "query_only_FALLBACK"

msa_il7ra = MSA.from_a3m(path=A3M_IL7RA)
query = msa_il7ra.entries[0].sequence
match_cols = sum(1 for ch in query if not is_a3m_insertion(ch))
q_plain = "".join(ch for ch in query if not is_a3m_insertion(ch)).replace("-", "")

print(f"\nMSA: depth={msa_il7ra.depth}  match_columns={match_cols}  (target={len(IL7RA)} aa)")
print(f"     header[0]={msa_il7ra.entries[0].header!r}")
if match_cols != len(IL7RA):
    REPORT["errors"].append({"step": "msa", "msg": f"match_columns {match_cols} != len(target) {len(IL7RA)}"})
    save_partial()
    raise SystemExit(
        f"DỪNG: a3m có {match_cols} match column nhưng target dài {len(IL7RA)} aa. "
        f"prepare_input sẽ lỗi index. Xoá {A3M_IL7RA} rồi chạy lại cell 6."
    )
if q_plain != IL7RA:
    print("CẢNH BÁO: query trong a3m khác sequence target (có thể do '-'), vẫn chạy tiếp nhưng ghi lại.")

msa_realistic = msa_il7ra.depth >= 50
print(f"     msa_depth_realistic = {msa_realistic}  "
      f"({'ok' if msa_realistic else 'KHÔNG — đây không phải tái dựng cửa sàng'})")

REPORT["steps"]["msa"] = {
    "source": msa_source, "a3m_path": A3M_IL7RA,
    "depth": msa_il7ra.depth, "match_columns": match_cols,
    "query_matches_target": q_plain == IL7RA,
    "msa_depth_realistic": bool(msa_realistic),
    "fetch_info": msa_info,
    "note": ("MSA thật từ ColabFold uniref" if msa_source.startswith("colabfold")
             else "a3m chỉ có query: CHỈ chứng minh code path, KHÔNG phải cấu hình cửa sàng"),
}
save_partial()
budget("(cell 6 xong)")

## Cell 7 — adapter ESMFold2 → ipSAE (câu hỏi 5), và chỗ "gotcha" thứ tự token

Research cảnh báo: `result.pae` ở **thứ tự token** của `prepare_input()`, **không** phải thứ tự
residue trong mmCIF. Sai chỗ này thì mọi điểm đều sai mà không báo lỗi.

Cách xử lý ở đây — đọc từ source `esm/models/esmfold2/{prepare_input,output}.py`:

1. `prepare_input()` gán `token_index` tăng dần theo đúng thứ tự `input.sequences`, và
   `build_molecular_complex_from_features()` lặp đúng theo thứ tự `chain_infos` đó. Với input
   **chỉ gồm protein, không modification, không ligand**, thứ tự token **chính là** thứ tự residue.
2. Ta **không tin** điều đó mà **tự dựng permutation** từ `chain_infos[*].tokens[*].token_index`
   (rẻ, chạy trên CPU) rồi **assert** nó là identity. Nếu không phải identity, notebook áp dụng
   permutation và ghi cờ `token_permutation_identity: false`.
3. Ta **tự ghi file PDB theo đúng thứ tự token** (thay vì dùng `to_mmcif()` của biotite), rồi dùng
   **đường AF2 của `ipsae.py`** (`.pdb` + `.json` chứa `pae`/`plddt`/`iptm`/`ptm`). Đường AF2 dùng
   `pae` nguyên matrix, không có bước `token_mask` của đường AF3 → **bớt một chỗ có thể sai âm thầm**.
   Adapter này đã được test offline với PDB tổng hợp: `ipsae.py` chạy rc=0 và ra bảng đúng format.

In [ ]:
# ============================== CELL 7: ipSAE ADAPTER ==============================
budget("(bắt đầu cell 7)")
import hashlib, numpy as np, urllib.request

UA = globals().get("UA", "proteinAI-changjiangcup/0.1 (smoke test)")

# ---- lấy ipsae.py v1 (DunbrackLab/IPSAE), ghi lại sha256 để tái lập được ----
IPSAE_PY = f"{D_DATA}/ipsae.py"
IPSAE_URL = "https://raw.githubusercontent.com/DunbrackLab/IPSAE/main/ipsae.py"
if not os.path.exists(IPSAE_PY):
    print(f"Tải {IPSAE_URL}")
    req = urllib.request.Request(IPSAE_URL, headers={"User-Agent": UA})
    with urllib.request.urlopen(req, timeout=60) as r, open(IPSAE_PY, "wb") as f:
        f.write(r.read())
IPSAE_SHA = hashlib.sha256(open(IPSAE_PY, "rb").read()).hexdigest()
print(f"ipsae.py: {IPSAE_PY}\n  sha256 = {IPSAE_SHA}")

PAE_CUTOFF, DIST_CUTOFF = 10, 10     # giả định cửa sàng (research: 10/10, variant d0res)

# ---- audit thứ tự token (chạy trên CPU, không tốn GPU) ----
def token_order_audit(spi):
    """Dựng permutation mmCIF-order -> PAE-token-index từ chain_infos. Trả về (perm, info)."""
    _feats, chain_infos = builder.prepare_input(spi, seed=0)   # device=None -> CPU
    perm, chain_of, resnum_of, resname_of = [], [], [], []
    for ci in chain_infos:
        seen = {}
        for tk in ci.tokens:
            if tk.residue_index in seen:      # residue nhiều token (ligand/PTM) — ta không có
                continue
            seen[tk.residue_index] = True
            perm.append(tk.token_index)
            chain_of.append(ci.chain_id)
            resnum_of.append(len(seen))
            resname_of.append(tk.residue_name)
    perm = np.asarray(perm, dtype=np.int64)
    n_tok = sum(len(ci.tokens) for ci in chain_infos)
    info = {
        "n_tokens": int(n_tok), "n_residues_out": int(perm.size),
        "is_permutation": bool(np.array_equal(np.sort(perm), np.arange(perm.size))),
        "is_identity": bool(perm.size == n_tok and np.array_equal(perm, np.arange(perm.size))),
        "chain_order": [ci.chain_id for ci in chain_infos],
        "chain_lengths": [len(ci.tokens) for ci in chain_infos],
    }
    del _feats, chain_infos
    return perm, chain_of, resnum_of, resname_of, info

# ---- tự ghi PDB theo đúng thứ tự token (cột cố định, khớp parse_pdb_atom_line của ipsae) ----
def write_pdb_token_order(mc, path):
    """mc = result.complex. Residue trong file xếp theo đúng thứ tự token của model.
    Trả về (n_residues, n_ca, n_cb_or_glyca, chain_ids_per_residue)."""
    chain_lookup = mc.metadata.chain_lookup
    lines, serial = [], 1
    n_ca = n_cb = 0
    chain_ids, res_counter = [], {}
    for tok_idx, (s, e) in enumerate(mc.token_to_atoms):
        resname = str(mc.sequence[tok_idx])[:3]
        cid_num = int(mc.chain_id[tok_idx])
        chain = str(chain_lookup[cid_num])[:1] or "A"
        res_counter[chain] = res_counter.get(chain, 0) + 1
        resnum = res_counter[chain]
        chain_ids.append(chain)
        has_ca = has_cb = False
        for a in range(int(s), int(e)):
            an = str(mc.atom_names[a]).strip().upper()
            el = str(mc.atom_elements[a]).strip().upper()[:2]
            x, y, z = (float(v) for v in mc.atom_positions[a])
            an4 = f" {an:<3}" if len(an) < 4 else an[:4]
            b = float(mc.plddt[tok_idx]) * 100.0
            lines.append(
                f"ATOM  {serial:5d} {an4} {resname:>3s} {chain:1s}{resnum:4d}    "
                f"{x:8.3f}{y:8.3f}{z:8.3f}{1.00:6.2f}{b:6.2f}          {el:>2s}"
            )
            serial += 1
            if an == "CA":
                has_ca = True
                if resname == "GLY":
                    has_cb = True
            if an == "CB":
                has_cb = True
        n_ca += has_ca; n_cb += has_cb
    lines += ["TER", "END"]
    with open(path, "w") as f:
        f.write("\n".join(lines) + "\n")
    return len(mc.token_to_atoms), n_ca, n_cb, chain_ids

def run_ipsae(result, tag, outdir, perm=None):
    """Ghi PDB + JSON kiểu AF2, chạy ipsae.py, parse bảng. Trả về dict."""
    os.makedirs(outdir, exist_ok=True)
    pdb_p  = f"{outdir}/{tag}.pdb"
    json_p = f"{outdir}/{tag}.json"

    n_res, n_ca, n_cb, chain_ids = write_pdb_token_order(result.complex, pdb_p)

    pae = result.pae.float().numpy()
    if perm is not None and not np.array_equal(perm, np.arange(perm.size)):
        pae = pae[np.ix_(perm, perm)]          # đưa PAE về đúng thứ tự residue trong PDB
    plddt = result.plddt.float().numpy().ravel()
    if perm is not None and not np.array_equal(perm, np.arange(perm.size)):
        plddt = plddt[perm]
    if plddt.max() <= 1.0:
        plddt = plddt * 100.0                  # ESMFold2 trả 0–1; ipsae/pDockQ cần 0–100

    assert pae.shape == (n_res, n_res), f"PAE {pae.shape} != ({n_res},{n_res})"
    assert n_ca == n_res, f"PDB có {n_ca} CA nhưng {n_res} residue"
    assert n_cb == n_res, f"PDB có {n_cb} CB(+GLY CA) nhưng {n_res} residue -> ipsae sẽ lệch shape"

    with open(json_p, "w") as f:
        json.dump({"pae": np.round(pae, 2).tolist(),
                   "plddt": np.round(plddt, 2).tolist(),
                   "iptm": float(result.iptm) if result.iptm is not None else -1.0,
                   "ptm":  float(result.ptm)  if result.ptm  is not None else -1.0}, f)

    r = subprocess.run([sys.executable, IPSAE_PY, json_p, pdb_p, str(PAE_CUTOFF), str(DIST_CUTOFF)],
                       capture_output=True, text=True, timeout=900)
    txt_p = f"{outdir}/{tag}_{PAE_CUTOFF:02d}_{DIST_CUTOFF:02d}.txt"
    out = {"ipsae_returncode": r.returncode, "ipsae_stdout": r.stdout[-1500:],
           "ipsae_stderr": r.stderr[-2500:], "txt": txt_p,
           "pdb": pdb_p, "json": json_p, "asym": {}, "max": {}}
    if r.returncode != 0 or not os.path.exists(txt_p):
        return out
    for line in open(txt_p):
        p = line.split()
        if len(p) < 6 or p[0] == "Chn1":
            continue
        try:
            val = float(p[5])
        except ValueError:
            continue
        key = f"{p[0]}->{p[1]}"
        if p[4] == "asym":
            out["asym"][key] = val
        elif p[4] == "max":
            out["max"][f"{p[0]}-{p[1]}"] = val
    if out["asym"]:
        out["ipsae_min"] = min(out["asym"].values())
        out["ipsae_max"] = max(out["asym"].values())
    return out

print(f"\nAdapter sẵn sàng. PAE/dist cutoff = {PAE_CUTOFF}/{DIST_CUTOFF}, variant d0res (cột 'ipSAE').")
REPORT["steps"]["ipsae_adapter"] = {
    "ipsae_py": IPSAE_PY, "ipsae_sha256": IPSAE_SHA, "ipsae_url": IPSAE_URL,
    "pae_cutoff": PAE_CUTOFF, "dist_cutoff": DIST_CUTOFF,
    "path": "AF2 (.pdb + .json with pae/plddt/iptm/ptm)",
    "pdb_written_in": "model token order (so PAE needs no permutation when identity holds)",
}
save_partial()
budget("(cell 7 xong)")

## Cell 8 — ba lần fold chính (câu hỏi 2 + 4 + 5)

| Run | Phức | MSA target | Mục đích |
|---|---|---|---|
| `A_gate_msa` | IL-7Rα + binder(+) | **có** | **đúng cấu hình cửa sàng** → VRAM, thời gian, ipSAE |
| `B_nomsa`    | IL-7Rα + binder(+) | không | tách riêng tác động của MSA |
| `C_neg_msa`  | IL-7Rα + binder(−) | **có** | negative control → có phân biệt được hay không |

Mỗi run ghi ngay vào Drive. Mỗi run in VRAM peak + phút + CU. Nếu run A đã ngốn quá nhiều
thì dừng tay — B và C là chẩn đoán, không bắt buộc.

Lưu ý: `prepare_input` sẽ in `UserWarning: No MSA provided for B, using single sequence mode`
cho chain binder. **Đó là đúng** — cửa sàng cố tình không cho binder MSA.

In [ ]:
# ============================== CELL 8: MAIN FOLDS ==============================
budget("(bắt đầu cell 8)")

def fold_once(tag, chains, num_loops, num_sampling_steps, seed=0, offload_esmc=False,
              chunk_size="default", run_ipsae_too=True, outdir=None):
    """chains = [(chain_id, sequence, msa_or_None), ...]. Trả về dict kết quả; không raise khi OOM."""
    outdir = outdir or D_RUNS
    total = sum(len(s) for _, s, _ in chains)
    rec = {"tag": tag, "total_residues": total,
           "chains": [{"id": c, "len": len(s), "msa_depth": (m.depth if m is not None else 0)}
                      for c, s, m in chains],
           "num_loops": num_loops, "num_sampling_steps": num_sampling_steps,
           "num_diffusion_samples": 1, "seed": seed,
           "lm_dropout": LM_DROPOUT, "msa_max_depth": MSA_MAX_DEPTH,
           "msa_column_mask_rate": MSA_COLUMN_MASK_RATE,
           "offload_esmc": bool(offload_esmc), "chunk_size": str(chunk_size),
           "ok": False}

    spi = StructurePredictionInput(sequences=[
        ProteinInput(id=c, sequence=s, msa=m) for c, s, m in chains
    ])

    # audit thứ tự token (CPU, rẻ)
    try:
        perm, _ch, _rn, _nm, tinfo = token_order_audit(spi)
        rec["token_order"] = tinfo
    except Exception as e:
        perm, rec["token_order"] = None, {"error": f"{type(e).__name__}: {e}"}

    model._offload_esmc = bool(offload_esmc)
    if chunk_size != "default":
        model.set_chunk_size(chunk_size)

    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    base = torch.cuda.memory_allocated() / 1024**3
    print(f"\n--- {tag}: {total} residue, loops={num_loops}, steps={num_sampling_steps}, "
          f"offload_esmc={offload_esmc}, chunk={chunk_size} ---", flush=True)
    t = time.time()
    result, err, oom = None, None, False
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("once")   # cho thấy 1 lần cảnh báo "No MSA provided for B"
            result = builder.fold(
                model, spi,
                num_loops=num_loops,
                num_sampling_steps=num_sampling_steps,
                num_diffusion_samples=1,
                seed=seed,
                # ba cái dưới ĐANG LÀ MẶC ĐỊNH của builder.fold() trong esm 3.4.3 —
                # viết ra để chúng xuất hiện trong báo cáo, vì chúng là thông số
                # cần hỏi organizer để tái dựng cửa sàng cho khớp.
                lm_dropout=LM_DROPOUT,
                msa_max_depth=MSA_MAX_DEPTH,
                msa_column_mask_rate=MSA_COLUMN_MASK_RATE,
            )
    except Exception as e:
        err = f"{type(e).__name__}: {e}"
        oom = ("out of memory" in err.lower()) or ("OutOfMemoryError" in err)
    wall = time.time() - t

    rec.update(wall_seconds=round(wall, 1),
               vram_baseline_gb=round(base, 3),
               vram_peak_gb=round(torch.cuda.max_memory_allocated() / 1024**3, 3),
               vram_peak_reserved_gb=round(torch.cuda.max_memory_reserved() / 1024**3, 3))
    rec["vram_activation_gb"] = round(rec["vram_peak_gb"] - base, 3)

    if err is not None:
        rec.update(ok=False, oom=oom, error=err[:1200])
        print(f"    {'OOM' if oom else 'LỖI'} sau {wall:.0f}s · peak {rec['vram_peak_gb']:.2f} GB")
        print(f"    {err[:400]}")
    else:
        rec.update(ok=True, oom=False,
                   plddt_mean=round(float(result.plddt.float().mean()), 4),
                   ptm=round(float(result.ptm), 4) if result.ptm is not None else None,
                   iptm=round(float(result.iptm), 4) if result.iptm is not None else None,
                   pae_shape=list(result.pae.shape) if result.pae is not None else None,
                   pae_dtype=str(result.pae.dtype) if result.pae is not None else None)
        print(f"    OK {wall:.0f}s ({wall/60:.1f} phút) · peak {rec['vram_peak_gb']:.2f} GB "
              f"(activation {rec['vram_activation_gb']:.2f} GB)")
        print(f"    pLDDT {rec['plddt_mean']:.3f} · pTM {rec['ptm']} · ipTM {rec['iptm']} "
              f"· PAE {rec['pae_shape']} {rec['pae_dtype']}")
        if run_ipsae_too:
            try:
                ip = run_ipsae(result, tag, outdir, perm=perm)
                rec["ipsae"] = ip
                if ip.get("asym"):
                    print(f"    ipSAE asym: " + "  ".join(f"{k}={v:.4f}" for k, v in ip["asym"].items()))
                    print(f"    ipSAE_min = {ip['ipsae_min']:.4f}   ipSAE_max = {ip['ipsae_max']:.4f}")
                else:
                    print(f"    ipSAE: KHÔNG PARSE ĐƯỢC (rc={ip['ipsae_returncode']})")
                    print(f"      stderr: {ip['ipsae_stderr'][:400]}")
            except Exception as e:
                rec["ipsae"] = {"error": f"{type(e).__name__}: {e}"}
                print(f"    ipSAE LỖI: {rec['ipsae']['error'][:400]}")

    # dọn sạch, kể cả sau OOM
    del result, spi, perm
    gc.collect(); gc.collect()
    model._offload_esmc = False
    if chunk_size != "default":
        model.set_chunk_size(64)      # _DEFAULT_CHUNK_SIZE trong esm 3.4.3
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    return rec


RUNS = [
    ("A_gate_msa", [("A", IL7RA, msa_il7ra), ("B", BINDER_POS, None)]),
    ("B_nomsa",    [("A", IL7RA, None),      ("B", BINDER_POS, None)]),
    ("C_neg_msa",  [("A", IL7RA, msa_il7ra), ("B", BINDER_NEG, None)]),
]
for tag, chains in RUNS:
    rec = fold_once(tag, chains, GATE_NUM_LOOPS, GATE_SAMPLING_STEPS, seed=0)
    REPORT["folds"].append(rec)
    save_partial()
    budget(f"(xong {tag})")

# so sánh dương tính vs âm tính
def _get(tag, key, default=None):
    for r in REPORT["folds"]:
        if r["tag"] == tag:
            return (r.get("ipsae") or {}).get(key, default) if key.startswith("ipsae_") else r.get(key, default)
    return default

pos = _get("A_gate_msa", "ipsae_min")
neg = _get("C_neg_msa", "ipsae_min")
print("\n================ KẾT LUẬN POSITIVE vs NEGATIVE CONTROL ================")
print(f"  binder đã validate (7S5B)  ipSAE_min = {pos}")
print(f"  binder đảo thứ tự          ipSAE_min = {neg}")
if isinstance(pos, float) and isinstance(neg, float):
    gap = pos - neg
    print(f"  khoảng cách = {gap:+.4f}")
    if pos < 0.5:
        print("  => BÁO ĐỘNG: binder THẬT cũng chỉ được điểm thấp. Setup chấm điểm của ta đáng nghi "
              "(MSA? cutoff? chiều ipSAE? thứ tự token?). KHÔNG tin bất kỳ design nào trước khi sửa.")
    elif gap < 0.15:
        print("  => BÁO ĐỘNG: pipeline KHÔNG phân biệt được thật/giả. Không dùng để chọn 20 sequence.")
    else:
        print("  => OK: pipeline phân biệt được. Đi tiếp được.")
    REPORT["steps"]["control_check"] = {"ipsae_min_positive": pos, "ipsae_min_negative": neg,
                                        "gap": round(gap, 4),
                                        "discriminates": bool(pos >= 0.5 and gap >= 0.15)}
else:
    print("  => không đủ số liệu để kết luận (xem lỗi ở trên).")
    REPORT["steps"]["control_check"] = {"ipsae_min_positive": pos, "ipsae_min_negative": neg,
                                        "discriminates": None}
save_partial()
budget("(cell 8 xong)")

## Cell 9 — probe TRẦN SỐ RESIDUE (câu hỏi 3) — cell quan trọng nhất

Maintainer của Biohub nói 24 GB "chịu được phức ~500 aa tổng". Chưa ai công bố số đo thật ở
bf16+fp8. Đây là lúc đo.

Leo thang bằng TL1A (180 aa/protomer) + binder:

| n protomer | + binder | tổng | ý nghĩa |
|---|---|---|---|
| 1 | 55 | 235 | TL1A monomer — chắc chắn phải chạy |
| 1 | 100 | 280 | monomer + binder dài nhất cuộc thi cho phép |
| 2 | 55 | 415 | nửa đường |
| **3** | **55** | **595** | **TL1A trimer sinh học + binder — câu hỏi thật** |
| **3** | **100** | **640** | trường hợp xấu nhất ta cần |
| 4 | 100 | 820 | đo dư địa |

Probe chạy `num_loops=4, num_sampling_steps=20` cho nhanh — **VRAM peak gần như độc lập với số
loop/step** vì chúng tuần tự và tái dùng bộ nhớ (đã đọc `_run_one_loop` trong source). Kích thước
lớn nhất **chạy được** sẽ được fold lại một lần ở đúng cấu hình cửa sàng (20/200) để xác nhận.

Khi OOM, probe thử lại **cùng kích thước đó** với hai đòn bẩy, theo thứ tự:
1. `model._offload_esmc = True` — đẩy ESMC (≈6.4 GB fp8) sang CPU sau khi tính LM features
2. thêm `model.set_chunk_size(32)` — hạ chunk của các phép L² (mặc định 64)

Dừng sau **2 kích thước liên tiếp fail (kể cả đã dùng đòn bẩy)**. Mỗi bước ghi xuống Drive ngay,
nên cell này chết cũng không mất số liệu các bước trước.

In [ ]:
# ============================== CELL 9: OOM CEILING PROBE ==============================
budget("(bắt đầu cell 9)")

PROBE = [
    (1, BINDER_POS),   # 235
    (1, BINDER_100),   # 280
    (2, BINDER_POS),   # 415
    (3, BINDER_POS),   # 595  <- TL1A trimer sinh học + binder
    (3, BINDER_100),   # 640  <- xấu nhất ta cần
    (4, BINDER_100),   # 820  <- dư địa
]
CHAIN_LETTERS = ["A", "B", "C", "D", "E"]

consecutive_fail, largest_ok = 0, None
for n_prot, binder in PROBE:
    total = n_prot * len(TL1A) + len(binder)
    tag = f"P_tl1a{n_prot}x180_b{len(binder)}_{total}aa"
    chains = [(CHAIN_LETTERS[i], TL1A, None) for i in range(n_prot)]
    chains.append((CHAIN_LETTERS[n_prot], binder, None))

    attempts = [
        {"offload_esmc": False, "chunk_size": "default"},
        {"offload_esmc": True,  "chunk_size": "default"},
        {"offload_esmc": True,  "chunk_size": 32},
    ]
    passed = None
    for k, kw in enumerate(attempts):
        rec = fold_once(f"{tag}_try{k}", chains, PROBE_NUM_LOOPS, PROBE_SAMPLING_STEPS,
                        seed=0, run_ipsae_too=False, **kw)
        rec["probe_n_protomers"] = n_prot
        rec["probe_binder_len"] = len(binder)
        REPORT["ceiling_probe"].append(rec)
        save_partial()
        budget(f"(probe {total} aa, lần thử {k+1}/{len(attempts)})")
        if rec["ok"]:
            passed = rec
            break
        if not rec.get("oom"):
            print("    (không phải OOM — không thử đòn bẩy nữa cho kích thước này)")
            break

    if passed:
        largest_ok = {"total_residues": total, "n_protomers": n_prot,
                      "binder_len": len(binder), "tag": passed["tag"],
                      "vram_peak_gb": passed["vram_peak_gb"],
                      "wall_seconds": passed["wall_seconds"],
                      "offload_esmc": passed["offload_esmc"],
                      "chunk_size": passed["chunk_size"]}
        consecutive_fail = 0
        print(f"  ==> {total} aa CHẠY ĐƯỢC (peak {passed['vram_peak_gb']:.2f} GB, "
              f"{passed['wall_seconds']:.0f}s, offload={passed['offload_esmc']}, "
              f"chunk={passed['chunk_size']})")
    else:
        consecutive_fail += 1
        print(f"  ==> {total} aa KHÔNG CHẠY (đã thử hết đòn bẩy)")
        if consecutive_fail >= 2:
            print("  Dừng probe: 2 kích thước liên tiếp fail.")
            break

print("\n================ TRẦN SỐ RESIDUE ================")
if largest_ok:
    print(f"  Lớn nhất chạy được: {largest_ok['total_residues']} aa "
          f"(peak {largest_ok['vram_peak_gb']:.2f} GB, offload_esmc={largest_ok['offload_esmc']}, "
          f"chunk={largest_ok['chunk_size']})")
else:
    print("  KHÔNG kích thước nào chạy được — xem lỗi ở trên.")

fails = [r for r in REPORT["ceiling_probe"] if r.get("oom")]
first_oom = min((r["total_residues"] for r in fails), default=None)
print(f"  OOM đầu tiên ở: {first_oom} aa" if first_oom else "  Không gặp OOM trong các kích thước đã thử.")

TRIMER_TOTAL = 3 * len(TL1A) + len(BINDER_100)   # 640
trimer_ok = bool(largest_ok and largest_ok["total_residues"] >= 3 * len(TL1A) + len(BINDER_POS))
print(f"\n  => TL1A trimer (3x180) + binder có khả thi trên L4 không? "
      f"{'CÓ' if trimer_ok else 'KHÔNG'}")
if not trimer_ok:
    print("     -> TL1A phải chấm theo monomer tại chỗ, và/hoặc đẩy phần trimer sang "
          "L40S/A100 thuê hoặc platform sponsor (MatwingsVenus / AsterFire).")

REPORT["steps"]["ceiling"] = {
    "largest_ok": largest_ok,
    "first_oom_total_residues": first_oom,
    "tl1a_trimer_plus_binder_feasible": trimer_ok,
    "probe_settings": {"num_loops": PROBE_NUM_LOOPS, "num_sampling_steps": PROBE_SAMPLING_STEPS},
    "note": "VRAM peak coi như độc lập với num_loops/num_sampling_steps (chúng tuần tự, tái dùng bộ nhớ)",
}
save_partial()

# xác nhận: fold lại kích thước lớn nhất chạy được ở ĐÚNG cấu hình cửa sàng (20/200)
if largest_ok and elapsed_min() < 0.75 * HARD_CAP_MIN:
    n_prot, blen = largest_ok["n_protomers"], largest_ok["binder_len"]
    binder = BINDER_POS if blen == len(BINDER_POS) else BINDER_100
    chains = [(CHAIN_LETTERS[i], TL1A, None) for i in range(n_prot)]
    chains.append((CHAIN_LETTERS[n_prot], binder, None))
    print(f"\nXác nhận lại {largest_ok['total_residues']} aa ở cấu hình cửa sàng "
          f"({GATE_NUM_LOOPS} loop / {GATE_SAMPLING_STEPS} step) ...")
    rec = fold_once(f"V_confirm_{largest_ok['total_residues']}aa_gatecfg", chains,
                    GATE_NUM_LOOPS, GATE_SAMPLING_STEPS, seed=0, run_ipsae_too=False,
                    offload_esmc=largest_ok["offload_esmc"],
                    chunk_size=(32 if largest_ok["chunk_size"] == "32" else "default"))
    REPORT["ceiling_probe"].append(rec)
    REPORT["steps"]["ceiling"]["confirm_at_gate_settings"] = {
        "ok": rec["ok"], "vram_peak_gb": rec["vram_peak_gb"],
        "wall_seconds": rec["wall_seconds"], "oom": rec.get("oom"),
    }
    save_partial()
else:
    print("\nBỏ qua bước xác nhận ở cấu hình cửa sàng (hết ngân sách thời gian hoặc không có kích thước nào chạy).")

budget("(cell 9 xong)")

## Cell 10 — BÁO CÁO

In block `PASTE CHO CLAUDE` **và** ghi `out/01_esmfold2_smoke_report.json` lên Drive.
Chọn cách nào tiện hơn: copy block text, hoặc tải file JSON gửi vào chat.

Chỗ `<<ĐIỀN TAY>>`: số **compute unit còn lại** (Colab hiện ở menu trên phải → "Compute units").
Điền trước-và-sau khi chạy nếu bạn nhớ, chỉ cần số sau cũng được.

In [ ]:
# ============================== CELL 10: REPORT ==============================
A = next((r for r in REPORT["folds"] if r["tag"] == "A_gate_msa"), {}) or {}
B = next((r for r in REPORT["folds"] if r["tag"] == "B_nomsa"), {}) or {}
C = next((r for r in REPORT["folds"] if r["tag"] == "C_neg_msa"), {}) or {}
ld = REPORT["steps"].get("load_model", {})
ce = REPORT["steps"].get("ceiling", {})
ms = REPORT["steps"].get("msa", {})

REPORT["answers"] = {
    "q1_esmfold2_runs_on_l4": bool(A.get("ok")),
    "q1_how": {
        "load_call": 'EsmFold2Model.from_pretrained(dir, device="cuda", dtype=torch.bfloat16, '
                     'esmc_precision="fp8", load_esmc=True)',
        "vram_weights_gb": ld.get("vram_weights_gb"),
        "load_minutes": round((ld.get("seconds") or 0) / 60.0, 1),
    },
    "q2_main_complex": {
        "total_residues": A.get("total_residues"),
        "vram_peak_gb": A.get("vram_peak_gb"),
        "vram_activation_gb": A.get("vram_activation_gb"),
        "wall_seconds": A.get("wall_seconds"),
        "settings": f"num_loops={GATE_NUM_LOOPS}, num_sampling_steps={GATE_SAMPLING_STEPS}, "
                    f"num_diffusion_samples=1, seed=0, lm_dropout={LM_DROPOUT}, "
                    f"msa_max_depth={MSA_MAX_DEPTH}, msa_column_mask_rate={MSA_COLUMN_MASK_RATE}, "
                    f"kernel_backend={ld.get('kernel_backend_effective')}, "
                    f"esmc_precision={ESMC_PRECISION}, dtype={MODEL_DTYPE}",
        "gpu_hours_per_1000_folds": (round(A["wall_seconds"] * 1000 / 3600.0, 1)
                                     if A.get("wall_seconds") else None),
        "cu_per_1000_folds": (round(A["wall_seconds"] * 1000 / 3600.0 * CU_PER_HOUR, 1)
                              if A.get("wall_seconds") else None),
    },
    "q3_residue_ceiling": {
        "largest_ok_total_residues": (ce.get("largest_ok") or {}).get("total_residues"),
        "first_oom_total_residues": ce.get("first_oom_total_residues"),
        "tl1a_trimer_plus_binder_feasible": ce.get("tl1a_trimer_plus_binder_feasible"),
        "levers_that_helped": sorted({
            ("offload_esmc" if r.get("offload_esmc") else None)
            for r in REPORT["ceiling_probe"] if r.get("ok")
        } - {None}) + sorted({
            ("chunk_size_32" if str(r.get("chunk_size")) == "32" else None)
            for r in REPORT["ceiling_probe"] if r.get("ok")
        } - {None}),
    },
    "q4_msa_path": {
        "msa_encoder_present": ld.get("msa_encoder_present"),
        "msa_source": ms.get("source"),
        "msa_depth": ms.get("depth"),
        "msa_depth_realistic": ms.get("msa_depth_realistic"),
        "fold_with_msa_ok": bool(A.get("ok")),
        "fold_without_msa_ok": bool(B.get("ok")),
        "iptm_with_msa": A.get("iptm"), "iptm_without_msa": B.get("iptm"),
        "ipsae_min_with_msa": (A.get("ipsae") or {}).get("ipsae_min"),
        "ipsae_min_without_msa": (B.get("ipsae") or {}).get("ipsae_min"),
        "honest_caveat": ("MSA thật (ColabFold uniref) — có thể so với cửa sàng"
                          if str(ms.get("source", "")).startswith("colabfold")
                          else "a3m CHỈ CÓ QUERY: chỉ chứng minh code path chạy, KHÔNG tái dựng cửa sàng"),
    },
    "q5_pae_to_ipsae": {
        "pae_shape": A.get("pae_shape"), "pae_dtype": A.get("pae_dtype"),
        "token_order_identity": (A.get("token_order") or {}).get("is_identity"),
        "token_order_is_valid_permutation": (A.get("token_order") or {}).get("is_permutation"),
        "ipsae_returncode": (A.get("ipsae") or {}).get("ipsae_returncode"),
        "ipsae_asym": (A.get("ipsae") or {}).get("asym"),
        "ipsae_max_row": (A.get("ipsae") or {}).get("max"),
        "ipsae_sha256": REPORT["steps"].get("ipsae_adapter", {}).get("ipsae_sha256"),
    },
    "control_check": REPORT["steps"].get("control_check"),
}
REPORT["finished_utc"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
REPORT["elapsed_min"] = round(elapsed_min(), 2)
REPORT["cu_estimate"] = round(cu_burned(), 3)

FINAL = f"{D_OUT}/01_esmfold2_smoke_report.json"
with open(FINAL, "w") as f:
    json.dump(REPORT, f, indent=2, default=str)

print("=" * 40)
print("PASTE CHO CLAUDE — BẮT ĐẦU")
print("=" * 40)
print(json.dumps(REPORT, indent=2, default=str, ensure_ascii=False))
print("=" * 40)
print("PASTE CHO CLAUDE — KẾT THÚC")
print("=" * 40)

print(f"\nĐã ghi: {FINAL}")
print(f"Artifact (PDB / PAE json / bảng ipSAE): {D_RUNS}")
print(f"Weights cache (ĐỪNG XOÁ): {D_W}")
print(f"\nTổng: {REPORT['elapsed_min']:.1f} phút · ước {REPORT['cu_estimate']:.2f} compute unit.")
print("\nCÒN MỘT VIỆC TAY: mở menu Colab góc trên phải, xem 'Compute units' còn lại,")
print("rồi sửa 2 chỗ <<ĐIỀN TAY>> trong JSON (hoặc nhắn số đó kèm khi gửi cho Claude).")